### **AI_AGENT_replyAI**

### **explore les autres fichiers**

In [ ]:
import pandas as pd
import json
import os

# Nouveau dataset
base_path = r'../Brave New World - train'

def load_json_robust(filename):
    full_path = os.path.join(base_path, filename)
    try:
        return pd.read_json(full_path)
    except:
        try:
            return pd.read_json(full_path, orient="records")
        except:
            with open(full_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            return pd.json_normalize(data)

# Chargement
df_transactions = pd.read_csv(os.path.join(base_path, "transactions.csv"))
df_users        = load_json_robust("users.json")
df_sms          = load_json_robust("sms.json")
df_mails        = load_json_robust("mails.json")
df_locations    = load_json_robust("locations.json")

print("✅ Fichiers chargés !")
print("Transactions:", df_transactions.shape)
print("Users:", df_users.shape)
print("SMS:", df_sms.shape)
print("Mails:", df_mails.shape)
print("Locations:", df_locations.shape)

print("\n--- TRANSACTIONS ---")
print(df_transactions.head(3))

print("\n--- USERS ---")
print(df_users.to_string())

print("\n--- COLONNES TX ---")
print(df_transactions.columns.tolist())

In [ ]:
base_path = r'../Brave New World - train'

print("--- HEAD TRANSACTIONS ---")
display(df_transactions.head())

print("\n--- HEAD USERS ---")
display(df_users.head())

print("\n--- HEAD SMS ---")
display(df_sms.head())

print("\n--- HEAD MAILS ---")
display(df_mails.head())

print("\n--- HEAD LOCATIONS ---")
display(df_locations.head())

# Infos critiques
print("\n--- USERS COMPLETS ---")
print(df_users[['first_name','last_name','salary',
    'job','iban','residence']].to_string())

print("\n--- TRANSACTIONS SANS DESCRIPTION ---")
print(df_transactions[
    df_transactions['description'].isna()
][['transaction_id','sender_id','recipient_id',
   'amount','timestamp']].to_string())

print("\n--- HEURES NOCTURNES ---")
df_transactions['hour'] = pd.to_datetime(
    df_transactions['timestamp']).dt.hour
print(df_transactions[
    df_transactions['hour'] < 6
][['transaction_id','sender_id','amount',
   'description','hour']].to_string())

In [ ]:
print("Transactions:", df_transactions.shape)
print("Users:", df_users.shape)
print("SMS:", df_sms.shape)
print("Mails:", df_mails.shape)
print("Locations:", df_locations.shape)

print("\n--- DESCRIPTIONS ---")
print(df_transactions['description'].value_counts().to_string())

print("\n--- PÉRIODE ---")
print("Début:", df_transactions['timestamp'].min())
print("Fin:", df_transactions['timestamp'].max())

print("\n--- TYPES DE TRANSACTIONS ---")
print(df_transactions['transaction_type'].value_counts())

print("\n--- MONTANTS ---")
print(df_transactions['amount'].describe())

# Transactions qui NE sont PAS salary/rent

In [ ]:
# Transactions qui NE sont PAS salary/rent
mask = ~df_transactions['description'].str.contains(
    'Salary|Rent', na=False
)
print("Transactions suspectes potentielles:")
print(df_transactions[mask][['transaction_id','sender_id',
    'recipient_id','amount','description','timestamp']])

# Comparer montants normaux vs suspects
print("\nMontant moyen salary/rent:", 
    df_transactions[~mask]['amount'].mean())
print("Montant moyen autres:", 
    df_transactions[mask]['amount'].mean())

### **donnation**

In [ ]:
print(df_transactions[
    df_transactions['description'] == 'Donation'
][['transaction_id','sender_id','amount',
   'timestamp']].to_string())

### **pishing**

In [ ]:
phishing_keywords = [
    'paypa1', 'amaz0n', 'verify', 'bit.ly', 
    'urgent', 'suspend', 'account lock', 'click here'
]

for i, row in df_sms.iterrows():
    sms_text = str(row['sms'])
    if any(k in sms_text.lower() for k in phishing_keywords):
        print(f"\n=== SMS SUSPECT {i} ===")
        print(sms_text[:300])
        print("---")

print("\n--- MAILS SUSPECTS ---")
for i, row in df_mails.iterrows():
    mail_text = str(row['mail'])
    if any(k in mail_text.lower() for k in phishing_keywords):
        print(f"\n=== MAIL SUSPECT {i} ===")
        print(mail_text[:300])
        print("---")

In [ ]:
for i, row in df_mails.iterrows():
    print(f"\n=== MAIL {i} ===")
    print(row['mail'][:300])
    print("---")

telephone


In [ ]:
# Extraire tous les numéros uniques des SMS
phones = set()
for i, row in df_sms.iterrows():
    sms_text = str(row['sms'])
    if 'To:' in sms_text:
        phone = sms_text.split('To:')[1].split('\n')[0].strip()
        phones.add(phone)

print("Numéros trouvés:")
for p in sorted(phones):
    print(p)

In [ ]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

base_path = r'../Brave New World - train'

# ===== 1. SMS SUSPECTS =====
print("=== SMS SUSPECTS ===")
phishing_keywords = [
    'paypa1', 'amaz0n', 'verify', 'bit.ly',
    'urgent', 'suspend', 'account lock',
    'netfl1x', 'renewal', 'click here'
]
for i, row in df_sms.iterrows():
    if any(x in str(row['sms']).lower() for x in phishing_keywords):
        print(f"\n--- SMS {i} ---")
        print(str(row['sms'])[:300])

# ===== 2. MAILS SUSPECTS =====
print("\n=== MAILS SUSPECTS ===")
for i, row in df_mails.iterrows():
    if any(x in str(row['mail']).lower() for x in phishing_keywords):
        print(f"\n--- MAIL {i} ---")
        print(str(row['mail'])[:400])
        print("---")

# ===== 3. SANS DESCRIPTION =====
print("\n=== SANS DESCRIPTION ===")
df_transactions['timestamp'] = pd.to_datetime(df_transactions['timestamp'])
df_transactions['hour'] = df_transactions['timestamp'].dt.hour
no_desc = df_transactions[df_transactions['description'].isna()]
print(no_desc[['transaction_id','sender_id','recipient_id',
               'amount','timestamp']].to_string())

# ===== 4. NOCTURNES =====
print("\n=== NOCTURNES ===")
night = df_transactions[df_transactions['hour'] < 6]
print(night[['transaction_id','sender_id','amount',
             'description','hour']].to_string())

# ===== 5. DESTINATAIRES INCONNUS =====
print("\n=== DESTINATAIRES INCONNUS ===")
known_ibans = df_users['iban'].tolist()
unknown = df_transactions[
    ~df_transactions['recipient_iban'].isin(known_ibans)
]
print(unknown[['transaction_id','recipient_id',
               'amount','description']].head(15).to_string())

# ===== 6. GPS CORRELATION =====
print("\n=== GPS SUSPECTS ===")
with open(f'{base_path}/locations.json') as f:
    loc_bnw = json.load(f)

df_loc_bnw = pd.DataFrame(loc_bnw)
df_loc_bnw['timestamp'] = pd.to_datetime(df_loc_bnw['timestamp'])

known_users_bnw = {
    'RTZO-JMIX-7D1-GAU-0': 'Jim',
    'VLNT-SLVS-7DC-PIO-0': 'Silvestro',
    'FCHN-VTSI-7FA-BRE-0': 'Vitus',
    'NNLL-BRTU-7D3-RAV-0': 'Uberto',
    'WTZL-TNIA-7D9-PFA-0': 'Tania',
    'JNSO-PLNA-7D0-FEL-1': 'Pauline',
    'CSPR-GRHL-800-ILM-1': 'Gerhild'
}

for _, tx in no_desc.iterrows():
    if tx['sender_id'] not in known_users_bnw:
        continue
    tx_time = tx['timestamp']
    diff = (df_loc_bnw['timestamp'] - tx_time).abs()
    nearby = df_loc_bnw[
        (df_loc_bnw['biotag'] == tx['sender_id']) &
        (diff < pd.Timedelta(hours=24))
    ]
    name = known_users_bnw[tx['sender_id']]
    print(f"\nTX: {tx['transaction_id'][:8]} | {name} | {tx['amount']}€")
    if len(nearby) == 0:
        print("  → GPS ABSENT ⚠️")
    else:
        print(f"  → GPS: {nearby['city'].values}")

# ===== 7. FEATURES + FRAUD SCORE =====
print("\n=== FEATURES + FRAUD SCORE ===")

suspicious_recipients_bnw = [
    'MKTBR25408', 'RENT75951', 'SHPRB44898',
    'PAYHR95905', 'TRDDN82305'
]

salary_map_bnw = {
    'RTZO-JMIX-7D1-GAU-0': 35200/12,
    'VLNT-SLVS-7DC-PIO-0': 48900/12,
    'FCHN-VTSI-7FA-BRE-0': 86600/12,
    'NNLL-BRTU-7D3-RAV-0': 31400/12,
    'WTZL-TNIA-7D9-PFA-0': 39800/12,
    'JNSO-PLNA-7D0-FEL-1': 41000/12,
    'CSPR-GRHL-800-ILM-1': 33600/12
}

regular_keywords_bnw = [
    'salary', 'rent', 'insurance', 'mobile',
    'internet', 'rideshare', 'health',
    'subscription', 'phone bill', 'loan payment'
]

df = df_transactions.copy()

# Features
df['f_no_description'] = df['description'].isna().astype(int)
df['f_night'] = (df['hour'] < 6).astype(int)
df['f_unknown_recipient'] = df['recipient_id'].isin(
    suspicious_recipients_bnw).astype(int)
df['f_null_recipient'] = df['recipient_id'].isna().astype(int)

df['monthly_salary'] = df['sender_id'].map(salary_map_bnw)
desc_str = df['description'].fillna('').str.lower()
is_regular = desc_str.apply(
    lambda d: any(k in d for k in regular_keywords_bnw))
df['f_amount_suspicious'] = (
    (df['monthly_salary'].notna()) &
    (df['amount'] > df['monthly_salary']) &
    (~is_regular)
).astype(int)

# GPS absent
def check_gps_bnw(tx):
    if tx['sender_id'] not in known_users_bnw:
        return 0
    diff = (df_loc_bnw['timestamp'] - tx['timestamp']).abs()
    nearby = df_loc_bnw[
        (df_loc_bnw['biotag'] == tx['sender_id']) &
        (diff < pd.Timedelta(hours=24))
    ]
    return 0 if len(nearby) > 0 else 1

df['f_gps_absent'] = df.apply(check_gps_bnw, axis=1)

features = [
    'f_no_description', 'f_night',
    'f_unknown_recipient', 'f_null_recipient',
    'f_amount_suspicious', 'f_gps_absent'
]

df['fraud_score'] = df[features].sum(axis=1)

print("\n=== SUSPECTS FINAUX ===")
suspects_df = df[df['fraud_score'] >= 1].sort_values(
    'fraud_score', ascending=False)
print(suspects_df[['transaction_id','sender_id',
    'amount','hour','fraud_score'] + features].to_string())

# ===== 8. HEATMAP =====
df_suspects = suspects_df.copy()
df_suspects['tx_short'] = df_suspects['transaction_id'].str[:8]

feature_labels = {
    'f_no_description': 'No Description',
    'f_night': 'Night',
    'f_unknown_recipient': 'Unknown Recipient',
    'f_null_recipient': 'Null Recipient',
    'f_amount_suspicious': 'High Amount',
    'f_gps_absent': 'GPS Absent'
}

heatmap_data = df_suspects.set_index('tx_short')[features]
heatmap_data.columns = list(feature_labels.values())
scores = df_suspects.set_index('tx_short')['fraud_score']

fig, ax = plt.subplots(figsize=(12, max(6, len(df_suspects)*0.4)))

sns.heatmap(
    heatmap_data, annot=True, fmt='d',
    cmap='RdYlGn_r', vmin=0, vmax=1,
    linewidths=0.5, linecolor='gray', ax=ax,
    cbar_kws={'label': '0=Normal 1=Suspect', 'shrink': 0.8}
)

ax.set_title(
    'Fraud Detection Heatmap — Brave New World\nReply Mirror 2026',
    fontsize=14, fontweight='bold', pad=20)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

for i, (tx, score) in enumerate(scores.items()):
    ax.text(len(features)+0.3, i+0.5,
            f'Score:{int(score)}', va='center',
            fontsize=9, fontweight='bold',
            color='red' if score >= 2 else 'orange')

n_high = len(df_suspects[df_suspects['fraud_score'] >= 2])
if n_high > 0:
    ax.axhline(y=n_high, color='red', linewidth=2,
               linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('fraud_heatmap_bnw.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Heatmap sauvegardée : fraud_heatmap_bnw.png")

### **tools**

In [ ]:
import pandas as pd
import json
import numpy as np
from collections import defaultdict

known_users = {
    'RTZO-JMIX-7D1-GAU-0': {'name': 'Jim', 'salary': 35200/12, 'city': 'Gautier',
                              'phone': '+11578090017', 'mail': 'jim.ortiz'},
    'VLNT-SLVS-7DC-PIO-0': {'name': 'Silvestro', 'salary': 48900/12, 'city': 'Piossasco',
                              'phone': '+17162281941', 'mail': 'silvestro.valentino'},
    'FCHN-VTSI-7FA-BRE-0': {'name': 'Vitus', 'salary': 86600/12, 'city': 'Bremen',
                              'phone': '+18127332895', 'mail': 'vitus.fechner'},
    'NNLL-BRTU-7D3-RAV-0': {'name': 'Uberto', 'salary': 31400/12, 'city': 'Ravenna',
                              'phone': '+14509736382', 'mail': 'uberto.iannelli'},
    'WTZL-TNAI-80B-PFA-0': {'name': 'Tania', 'salary': 39800/12, 'city': 'Pfarrkirchen',
                              'phone': '+15615579116', 'mail': 'tania.weitzel'},
    'JNSO-PLNA-7D0-FEL-1': {'name': 'Pauline', 'salary': 41000/12, 'city': 'Feltham',
                              'phone': '+19302780913', 'mail': 'pauline.jones'},
    'CSPR-GRHL-800-ILM-1': {'name': 'Gerhild', 'salary': 33600/12, 'city': 'Ilmenau',
                              'phone': '+17191618861', 'mail': 'gerhild.caspar'}
}

regular_senders = ['EMP68176','EMP40508','EMP98299','EMP47291','EMP81634','EMP52067','EMP73915']

regular_keywords = [
    'salary','rent','insurance','health','subscription','phone bill','loan payment',
    'professional liability','professional membership','creative suite','professional body',
    'consulting software','professional indemnity','creative cloud','coworking','adobe',
    'stock photo','creative tools','freelance content','internet subscription',
    'retiree health','senior health','home insurance','monthly health','private health',
    'monthly professional','indemnity','membership fee','professional dues','professional association'
]

CASH_TYPES = {'withdrawal', 'in-person payment'}

phishing_keywords = [
    'paypa1','amaz0n','verify','netfl1x','renewal','update payment','update card',
    'failed','suspended','inps-verify','pension-verify','pensi0n','socs','ch4se',
    'natw3st','chase-secure','hmrc-secure','paypa1-secure','amaz0n-verify','bremenbank'
]

USER_AMOUNT_STATS = {
    'RTZO-JMIX-7D1-GAU-0': {'mean': 132, 'std': 158},
    'VLNT-SLVS-7DC-PIO-0': {'mean': 70,  'std': 41},
    'FCHN-VTSI-7FA-BRE-0': {'mean': 526, 'std': 703},
    'NNLL-BRTU-7D3-RAV-0': {'mean': 208, 'std': 178},
    'WTZL-TNAI-80B-PFA-0': {'mean': 160, 'std': 125},
    'JNSO-PLNA-7D0-FEL-1': {'mean': 207, 'std': 204},
    'CSPR-GRHL-800-ILM-1': {'mean': 173, 'std': 225}
}

RECIPIENT_FREQUENCY = {
    'PROP93151':13,'RENT75951':13,'HOME28625':13,'DOM58085':12,'HOME36805':12,
    'HOME96755':12,'ABIT73300':12,'COMRB39809':5,'PAYPH11103':5,'VNDMR15938':3,
    'ENTSR71727':3,'PAYBR30021':3,'PAYWT22183':2,'SHPRB44898':2,'TRDDN82305':2,
    'PAYNV92614':2,'PAYDN65923':2,'PAYBR92660':2,'PAYCL92396':2,'SYSTR98866':2,
    'ACCRN27991':2,'BILMN49918':2,'MKTSW37100':2,
}

_tx_history = defaultdict(list)

def is_regular(tx):
    desc = str(tx.get('description','')).lower()
    sender = str(tx.get('sender_id',''))
    if sender in regular_senders: return True
    return any(k in desc for k in regular_keywords)

def is_cash(tx):
    return str(tx.get('transaction_type','')) in CASH_TYPES

def get_zscore(tx):
    sender = tx.get('sender_id','')
    if sender not in USER_AMOUNT_STATS: return 0.0
    s = USER_AMOUNT_STATS[sender]
    if s['std'] == 0: return 0.0
    return (float(tx.get('amount',0)) - s['mean']) / s['std']

def pre_filter(tx, score_result):
    amount = float(tx.get('amount',0))
    desc = str(tx.get('description',''))
    tx_type = str(tx.get('transaction_type',''))
    signals = score_result['signals']
    hint = score_result['hint']
    hour = pd.to_datetime(tx['timestamp']).hour
    zscore = get_zscore(tx)

    no_desc = desc in ('nan','None','') or pd.isna(tx.get('description'))
    null_recip = not tx.get('recipient_id') or str(tx.get('recipient_id'))=='nan'

    if is_regular(tx): return "normal","regular"
    if is_cash(tx):
        if amount > 500 and hour < 6: return "uncertain","cash_night_high"
        return "normal","cash"

    # HIGH VALUE → Agent3 direct
    if amount > 800 and not is_regular(tx) and not is_cash(tx):
        if no_desc or zscore > 2.0 or len(signals) >= 2:
            return "high_value_priority", f"priority_{int(amount)}EUR_z{zscore:.1f}"

    # Fraudes certaines
    if tx_type=='transfer' and no_desc and null_recip:
        return "fraud","transfer_no_desc_null_recipient"
    if tx_type=='transfer' and amount > 2000 and no_desc:
        return "fraud",f"critical_{int(amount)}EUR_no_desc"
    if zscore > 3.5 and not is_regular(tx):
        return "high_value_priority",f"zscore_extreme_{zscore:.1f}"

    if not signals and hint=='low_risk': return "normal","no_signals"
    return "uncertain",f"s:{len(signals)}_h:{hint}"

def check_rules(tx):
    signals, score = [], 0.0
    desc = str(tx.get('description',''))
    tx_type = str(tx.get('transaction_type',''))
    amount = float(tx.get('amount',0))
    hour = pd.to_datetime(tx['timestamp']).hour
    recipient = str(tx.get('recipient_id',''))
    zscore = get_zscore(tx)
    regular, cash = is_regular(tx), is_cash(tx)

    if not tx.get('description') or pd.isna(tx.get('description')):
        if tx_type=='transfer': score+=0.5; signals.append("missing_desc_transfer")
        elif not cash: score+=0.15; signals.append("missing_desc")

    if hour<6 and not cash and tx_type=='transfer': score+=0.3; signals.append("night_transfer")
    elif hour<6 and not cash: score+=0.1; signals.append("night_tx")

    if not tx.get('recipient_id') or str(tx.get('recipient_id'))=='nan':
        if tx_type=='transfer': score+=0.4; signals.append("null_recip_transfer")

    if zscore>3.0 and not regular: score+=0.4; signals.append(f"zscore_outlier_{zscore:.1f}")
    elif zscore>2.0 and not regular: score+=0.2; signals.append(f"zscore_high_{zscore:.1f}")

    freq = RECIPIENT_FREQUENCY.get(recipient, 0)
    if freq==0 and recipient not in ('nan','None','') and recipient and tx_type=='transfer' and not regular:
        score+=0.2; signals.append("unknown_recip_first_time")
    elif freq==1 and tx_type=='transfer' and amount>200:
        score+=0.1; signals.append("rare_recip_high_amount")

    dl = desc.lower()
    if 'donation' in dl and not regular and amount>200: score+=0.2; signals.append("large_donation")
    if 'emergency fund' in dl: score+=0.3; signals.append("emergency_fund")
    if 'savings transfer' in dl and amount>1000: score+=0.3; signals.append("large_savings_transfer")

    if amount>2000 and not regular and not cash: score+=0.5; signals.append(f"critical_{int(amount)}EUR")
    elif amount>500 and not regular and not cash and tx_type=='transfer': score+=0.15; signals.append("high_amount_transfer")

    return {"score":min(round(score,2),1.0),"signals":signals,
            "hint":"high_risk" if score>=0.5 else "suspicious" if score>=0.2 else "low_risk"}

def check_phishing(tx, sms_list, mails_list):
    sender = tx.get('sender_id','')
    if is_regular(tx) or is_cash(tx) or sender not in known_users:
        return {"score":0,"signals":[],"hint":"low_risk"}
    user = known_users[sender]
    signals, score = [], 0.0
    tx_time = pd.to_datetime(tx['timestamp'])

    for item in sms_list:
        t = str(item) if not isinstance(item,dict) else str(item.get('sms',''))
        if user['phone'] not in t: continue
        if any(k in t.lower() for k in phishing_keywords):
            try:
                ds = t.split('Date:')[1].split('\n')[0].strip()
                st = pd.to_datetime(ds)
                d = (tx_time-st).days
                if 0<=d<=30: score+=0.4; signals.append(f"phishing_sms_{d}d"); break
            except: continue

    for item in mails_list:
        t = str(item) if not isinstance(item,dict) else str(item.get('mail',''))
        if user['mail'] not in t.lower(): continue
        if any(k in t.lower() for k in phishing_keywords):
            score+=0.3; signals.append("phishing_mail"); break

    return {"score":min(round(score,2),1.0),"signals":signals,
            "hint":"high_risk" if score>=0.4 else "suspicious" if score>0 else "low_risk"}

def check_gps(tx, locations):
    sender = tx.get('sender_id','')
    if sender not in known_users or is_regular(tx) or is_cash(tx):
        return {"score":0,"signals":[],"hint":"low_risk"}
    tx_time = pd.to_datetime(tx['timestamp'])
    df_loc = pd.DataFrame(locations)
    df_loc['timestamp'] = pd.to_datetime(df_loc['timestamp'])
    diff = (df_loc['timestamp']-tx_time).abs()
    nearby = df_loc[(df_loc['biotag']==sender)&(diff<pd.Timedelta(hours=24))]
    signals, score = [], 0.0
    if len(nearby)==0: score+=0.1; signals.append("no_gps")
    else:
        cities = nearby['city'].unique()
        if known_users[sender]['city'] not in cities:
            score+=0.4; signals.append(f"gps_mismatch_{cities[0]}")
    return {"score":min(round(score,2),1.0),"signals":signals,
            "hint":"high_risk" if score>=0.4 else "suspicious" if score>0 else "low_risk"}

def check_user_profile(tx):
    sender = tx.get('sender_id','')
    if sender not in known_users or is_regular(tx) or is_cash(tx):
        return {"score":0,"signals":[],"hint":"low_risk"}
    user = known_users[sender]
    amount = float(tx.get('amount',0))
    signals, score = [], 0.0
    if amount>user['salary']*2: score+=0.4; signals.append(f"exceeds_2x_{round(user['salary'])}EUR")
    elif amount>user['salary']: score+=0.2; signals.append(f"exceeds_salary_{round(user['salary'])}EUR")
    return {"score":min(round(score,2),1.0),"signals":signals,
            "hint":"high_risk" if score>=0.4 else "suspicious" if score>0 else "low_risk"}

def check_burst(tx):
    sender = tx.get('sender_id','')
    if sender not in known_users or is_cash(tx):
        return {"score":0,"signals":[],"hint":"low_risk"}
    tx_time = pd.to_datetime(tx['timestamp'])
    signals, score = [], 0.0
    for pt in _tx_history[sender][-3:]:
        dm = abs((tx_time-pt).total_seconds()/60)
        if dm<10: score+=0.3; signals.append(f"burst_{dm:.0f}min"); break
    _tx_history[sender].append(tx_time)
    return {"score":min(round(score,2),1.0),"signals":signals,
            "hint":"high_risk" if score>=0.3 else "low_risk"}

def score_transaction(tx, sms_list, mails_list, locations):
    r1=check_rules(tx); r2=check_phishing(tx,sms_list,mails_list)
    r3=check_gps(tx,locations); r4=check_user_profile(tx); r5=check_burst(tx)
    all_signals = r1['signals']+r2['signals']+r3['signals']+r4['signals']+r5['signals']
    combined = round(r1['score']*0.40+r2['score']*0.20+r3['score']*0.15+r4['score']*0.15+r5['score']*0.10,2)
    hint = "high_risk" if combined>=0.4 else "suspicious" if combined>=0.15 else "low_risk"
    amount = float(tx.get('amount',0))
    if amount>2000 and not is_regular(tx) and not is_cash(tx): hint="high_risk"
    return {"combined_score":combined,"signal_count":len(all_signals),
            "signals":all_signals,"hint":hint,"amount":amount,"zscore":get_zscore(tx),
            "details":{"rules":r1,"phishing":r2,"gps":r3,"profile":r4,"burst":r5}}

### agent


In [ ]:
import os
import json
import re
import asyncio
import pandas as pd
import ulid
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langfuse import Langfuse, observe
from langfuse.langchain import CallbackHandler

load_dotenv()

# ===== MODÈLES PUISSANTS =====
model_a1 = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="openai/gpt-4o",
    temperature=0,
    max_tokens=120
)

model_a2 = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="openai/gpt-4o",
    temperature=0,
    max_tokens=150
)

model_a3 = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="anthropic/claude-sonnet-4",
    temperature=0,
    max_tokens=250
)

langfuse_client = Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host=os.getenv("LANGFUSE_HOST")
)

known_users_local = {
    'RTZO-JMIX-7D1-GAU-0': {'name':'Jim','salary':35200/12,'city':'Gautier'},
    'VLNT-SLVS-7DC-PIO-0': {'name':'Silvestro','salary':48900/12,'city':'Piossasco'},
    'FCHN-VTSI-7FA-BRE-0': {'name':'Vitus','salary':86600/12,'city':'Bremen'},
    'NNLL-BRTU-7D3-RAV-0': {'name':'Uberto','salary':31400/12,'city':'Ravenna'},
    'WTZL-TNAI-80B-PFA-0': {'name':'Tania','salary':39800/12,'city':'Pfarrkirchen'},
    'JNSO-PLNA-7D0-FEL-1': {'name':'Pauline','salary':41000/12,'city':'Feltham'},
    'CSPR-GRHL-800-ILM-1': {'name':'Gerhild','salary':33600/12,'city':'Ilmenau'}
}

def parse_json_safe(text: str) -> dict:
    text = text.strip().replace("```json","").replace("```","")
    match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if match:
        try: return json.loads(match.group())
        except: pass
    try: return json.loads(text)
    except: return {}

# ===== AGENT 1 : SCREENER STRICT =====
async def agent1_async(tx: dict, score_result: dict, session_id: str) -> dict:
    amount = float(tx.get('amount', 0))
    desc = str(tx.get('description', ''))
    hour = pd.to_datetime(tx['timestamp']).hour
    zscore = score_result.get('zscore', 0)

    prompt = f"""You are a strict bank fraud detection agent.

## TRANSACTION
```json
{{
  "type": "{tx.get('transaction_type')}",
  "amount_eur": {int(amount)},
  "hour": {hour},
  "description": "{desc[:60]}",
  "recipient_id": "{tx.get('recipient_id')}",
  "risk_score": {score_result['combined_score']},
  "z_score": {zscore:.2f},
  "signals": {json.dumps(score_result['signals'])}
}}
```

## RULES (apply strictly)
FRAUD if:
- transfer + missing_description + null_recipient → certain fraud
- burst_transaction signal detected
- phishing_sms signal + amount > 100EUR
- night_transfer + missing_description
- z_score > 2.5 + missing_description OR unknown_recipient

NORMAL if:
- withdrawal or in-person payment
- risk_score < 0.15 with no strong signals
- single weak signal only

REQUIREMENT: Need AT LEAST 2 corroborating signals OR 1 certain signal to flag fraud.

## RESPONSE
Return ONLY valid JSON:
{{"fraud": 0, "reason": "explanation"}}"""

    try:
        response = await model_a1.ainvoke([HumanMessage(content=prompt)])
        result = parse_json_safe(response.content)
        fraud = int(result.get("fraud", 0))
        if fraud not in [0, 1]:
            fraud = 1 if score_result['combined_score'] >= 0.55 else 0
        return {"fraud": fraud, "reason": str(result.get("reason",""))[:100]}
    except:
        fraud = 1 if score_result['combined_score'] >= 0.55 else 0
        return {"fraud": fraud, "reason": f"fallback_{score_result['combined_score']}"}

# ===== AGENT 2 : VALIDATOR PRÉCIS =====
async def agent2_async(tx: dict, score_result: dict,
                       agent1_reason: str, session_id: str) -> dict:
    amount = float(tx.get('amount', 0))
    desc = str(tx.get('description', ''))
    hour = pd.to_datetime(tx['timestamp']).hour
    zscore = score_result.get('zscore', 0)
    sender = tx.get('sender_id','')
    user_info = known_users_local.get(sender, {'name':'unknown','salary':0,'city':'unknown'})

    prompt = f"""You are a senior fraud analyst. Confirm or reject Agent1's fraud flag with high precision.

## FLAGGED TRANSACTION
```json
{{
  "type": "{tx.get('transaction_type')}",
  "amount_eur": {int(amount)},
  "hour": {hour},
  "description": "{desc[:60]}",
  "recipient_id": "{tx.get('recipient_id')}",
  "balance_after_eur": {tx.get('balance_after')},
  "z_score": {zscore:.2f},
  "risk_score": {score_result['combined_score']},
  "user_name": "{user_info['name']}",
  "user_monthly_salary_eur": {round(user_info['salary'])},
  "user_city": "{user_info['city']}"
}}
```

## SIGNALS DETECTED
{json.dumps(score_result['signals'], indent=2)}

## AGENT1 SAID
"{agent1_reason}"

## VALIDATION CRITERIA
CONFIRM FRAUD if:
- 2+ strong corroborating signals
- amount > 500EUR + missing_description + unknown_recipient
- phishing within 30 days + any transaction signal
- z_score > 2.5 + context supports fraud

REJECT (normal) if:
- Single weak signal + plausible legitimate explanation
- Amount consistent with normal user spending + description present
- Agent1 reason seems insufficient

ECONOMIC RULE: Missing {int(amount)}EUR fraud costs {int(amount)}EUR. False positive costs only minor friction.
{"Bias toward FRAUD confirmation at this amount." if amount > 400 else "Require clear evidence at this amount."}

## RESPONSE
Return ONLY valid JSON:
{{"fraud": 0, "confidence": 0.85, "reason": "detailed_explanation"}}"""

    try:
        response = await model_a2.ainvoke([HumanMessage(content=prompt)])
        result = parse_json_safe(response.content)
        fraud = int(result.get("fraud", 0))
        confidence = float(result.get("confidence", 0.5))
        if fraud not in [0, 1]: fraud = 0
        confidence = max(0.0, min(1.0, confidence))
        needs_a3 = (confidence < 0.80 and amount > 350) or \
                   (fraud == 1 and confidence < 0.70)
        return {
            "fraud": fraud, "confidence": confidence,
            "reason": str(result.get("reason",""))[:120],
            "needs_agent3": needs_a3
        }
    except:
        fraud = 1 if score_result['combined_score'] >= 0.55 else 0
        return {"fraud": fraud, "confidence": 0.5,
                "reason": "fallback", "needs_agent3": False}

# ===== AGENT 3 : CHIEF FRAUD OFFICER =====
@observe()
def agent3_secure(tx: dict, score_result: dict,
                  prev_result: dict, session_id: str) -> dict:
    handler = CallbackHandler()
    amount = float(tx.get('amount', 0))
    desc = str(tx.get('description', ''))
    zscore = score_result.get('zscore', 0)
    sender = tx.get('sender_id','')
    user_info = known_users_local.get(sender,
        {'name':'unknown','salary':0,'city':'unknown'})

    prompt = f"""You are the Chief Fraud Officer. Make the FINAL irreversible decision.

## HIGH-VALUE TRANSACTION
```json
{{
  "type": "{tx.get('transaction_type')}",
  "amount_eur": {int(amount)},
  "hour": {pd.to_datetime(tx['timestamp']).hour},
  "description": "{desc[:80]}",
  "recipient_id": "{tx.get('recipient_id')}",
  "balance_after_eur": {tx.get('balance_after')},
  "z_score": {zscore:.3f},
  "risk_score": {score_result['combined_score']},
  "salary_multiple": {round(amount / max(user_info['salary'], 1), 2)}
}}
```

## USER PROFILE
```json
{{
  "name": "{user_info['name']}",
  "monthly_salary_eur": {round(user_info['salary'])},
  "home_city": "{user_info['city']}"
}}
```

## ALL SIGNALS
{json.dumps(score_result['signals'], indent=2)}

## PREVIOUS ANALYSIS
"{prev_result.get('reason','N/A')}"
Previous confidence: {prev_result.get('confidence', 'N/A')}

## DECISION FRAMEWORK

CERTAIN FRAUD (any ONE):
- Transfer + null_recipient + missing_description
- Z-score > 3.5
- Phishing SMS ≤7 days before + amount > 200EUR

PROBABLE FRAUD (need TWO or more):
- Z-score > 2.0
- Missing description on transfer
- First-time unknown recipient
- Burst transaction pattern
- Phishing SMS ≤30 days before
- Amount > 2x monthly salary ({round(user_info['salary']*2)}EUR)

NOT FRAUD:
- Withdrawal or in-person payment
- Regular bill keywords in description
- Known frequent recipient

## ECONOMIC IMPACT
{"🚨 CRITICAL VALUE — default to FRAUD if any ambiguity" if amount > 800 else "⚖️ SIGNIFICANT — require 2 signals minimum"}
Fraud loss if missed: {int(amount)} EUR
False positive cost: minor customer friction

## RESPONSE
Return ONLY this exact JSON:
{{
  "fraud": <0_or_1>,
  "confidence": <0.00_to_1.00>,
  "reason": "<list specific signals that drove decision>"
}}"""

    response = model_a3.invoke(
        [HumanMessage(content=prompt)],
        config={"callbacks": [handler],
                "metadata": {"langfuse_session_id": session_id}}
    )

    result = parse_json_safe(response.content)
    fraud = int(result.get("fraud", 0))
    confidence = float(result.get("confidence", 0.5))
    if fraud not in [0, 1]:
        fraud = prev_result.get('fraud', 0)
    return {
        "fraud": fraud,
        "confidence": max(0.0, min(1.0, confidence)),
        "reason": str(result.get("reason",""))[:200]
    }

# ===== BATCH PROCESSING =====
async def process_batch(batch, sms_list, mails_list, locations, session_id):
    results = []

    scored = []
    for tx_dict in batch:
        sr = score_transaction(tx_dict, sms_list, mails_list, locations)
        pf_dec, pf_reason = pre_filter(tx_dict, sr)
        scored.append((tx_dict, sr, pf_dec, pf_reason))

    certain, high_value, uncertain = [], [], []
    for tx_dict, sr, pf_dec, pf_reason in scored:
        if pf_dec == "fraud": certain.append((tx_dict, sr, pf_reason))
        elif pf_dec == "high_value_priority": high_value.append((tx_dict, sr, pf_reason))
        elif pf_dec == "uncertain": uncertain.append((tx_dict, sr))

    for tx_dict, sr, reason in certain:
        results.append({'transaction_id': tx_dict['transaction_id'],
                        'amount': tx_dict['amount'], 'confidence': 0.97,
                        'reason': reason, 'agent': 'prefilter'})

    for tx_dict, sr, reason in high_value:
        a3 = agent3_secure(tx_dict, sr, {'reason': reason, 'fraud': 1, 'confidence': 0.7}, session_id)
        if a3['fraud'] == 1 or a3['confidence'] >= 0.65:
            results.append({'transaction_id': tx_dict['transaction_id'],
                            'amount': tx_dict['amount'], 'confidence': a3['confidence'],
                            'reason': a3['reason'], 'agent': 'agent3_priority'})

    if not uncertain: return results

    a1_tasks = [agent1_async(tx, sr, session_id) for tx, sr in uncertain]
    a1_results = await asyncio.gather(*a1_tasks)

    needs_a2 = []
    for (tx_dict, sr), a1 in zip(uncertain, a1_results):
        if a1['fraud'] == 1:
            amount = float(tx_dict.get('amount', 0))
            if amount > 100:
                needs_a2.append((tx_dict, sr, a1))
            elif sr['combined_score'] >= 0.40:
                results.append({'transaction_id': tx_dict['transaction_id'],
                                'amount': tx_dict['amount'], 'confidence': 0.72,
                                'reason': a1['reason'], 'agent': 'agent1'})

    if not needs_a2: return results

    a2_tasks = [agent2_async(tx, sr, a1['reason'], session_id) for tx, sr, a1 in needs_a2]
    a2_results = await asyncio.gather(*a2_tasks)

    needs_a3 = []
    for (tx_dict, sr, _), a2 in zip(needs_a2, a2_results):
        if a2.get('needs_agent3'):
            needs_a3.append((tx_dict, sr, a2))
        elif a2['fraud'] == 1 and a2['confidence'] >= 0.72:
            results.append({'transaction_id': tx_dict['transaction_id'],
                            'amount': tx_dict['amount'], 'confidence': a2['confidence'],
                            'reason': a2['reason'], 'agent': 'agent2'})

    for tx_dict, sr, a2 in needs_a3:
        a3 = agent3_secure(tx_dict, sr, a2, session_id)
        if a3['fraud'] == 1 or a3['confidence'] >= 0.70:
            results.append({'transaction_id': tx_dict['transaction_id'],
                            'amount': tx_dict['amount'], 'confidence': a3['confidence'],
                            'reason': a3['reason'], 'agent': 'agent3'})

    return results

# ===== MAIN =====
async def run_pipeline_async(folder: str) -> str:
    import time
    start = time.time()
    session_id = f"NETIZEN-{ulid.new().str}"
    print(f"🚀 Session: {session_id}\n📁 {folder}\n")

    df_tx = pd.read_csv(f'{folder}/transactions.csv')
    with open(f'{folder}/sms.json') as f: sms_raw = json.load(f)
    with open(f'{folder}/mails.json') as f: mails_raw = json.load(f)
    with open(f'{folder}/locations.json') as f: locations = json.load(f)

    sms_list = [{'sms':s} if isinstance(s,str) else s for s in sms_raw]
    mails_list = [{'mail':m} if isinstance(m,str) else m for m in mails_raw]

    BATCH_SIZE = 12
    all_txs = [row.to_dict() for _, row in df_tx.iterrows()]
    batches = [all_txs[i:i+BATCH_SIZE] for i in range(0, len(all_txs), BATCH_SIZE)]
    print(f"📦 {len(all_txs)} tx → {len(batches)} batches × {BATCH_SIZE}\n")

    all_frauds = []
    for i, batch in enumerate(batches):
        elapsed = time.time() - start
        print(f"  Batch {i+1}/{len(batches)} | {elapsed:.0f}s | frauds:{len(all_frauds)}", end="\r")
        all_frauds.extend(await process_batch(batch, sms_list, mails_list, locations, session_id))

    langfuse_client.flush()
    elapsed = time.time() - start

    agents = {}
    for f in all_frauds: agents[f.get('agent','?')] = agents.get(f.get('agent','?'), 0) + 1

    print(f"\n\n{'='*60}")
    print(f"📊 RÉSULTATS ({elapsed:.0f}s) — {len(all_frauds)}/{len(all_txs)} ({100*len(all_frauds)/len(all_txs):.1f}%)")
    for agent, count in sorted(agents.items()):
        print(f"  {agent:22s}: {count}")

    sorted_frauds = sorted(all_frauds, key=lambda x: x['amount'], reverse=True)
    print(f"\n💰 Valeur protégée : {sum(f['amount'] for f in all_frauds):.0f} EUR")
    print(f"\n🔍 FRAUDES DÉTECTÉES:")
    for f in sorted_frauds:
        print(f"  {f['transaction_id'][:8]} | {f['amount']:>8.0f}€ | "
              f"conf:{f['confidence']:.2f} | [{f['agent']:20s}] | {f['reason'][:55]}")

    with open('output_train.txt', 'w') as f:
        for fraud in all_frauds: f.write(fraud['transaction_id'] + '\n')

    print(f"\n✅ {len(all_frauds)} fraudes → output_train.txt")
    print(f"🔑 {session_id}")
    return session_id

# LANCER (Jupyter)
session_id = await run_pipeline_async("../Brave New World - validation")

In [ ]:
%%writefile tools_v4.py
# COLLE ICI TOUT LE CONTENU DE TA CELLULE TOOLS

In [ ]:
%%writefile agents_final_v2.py
# COLLE ICI TOUT LE CONTENU DE TA CELLULE AGENTS